# Neural Networks Capstone Project — Fixed Version

This notebook implements the two required NLP models:

1. **Slogan generator** — an LSTM that takes an industry category and predicts a slogan **one word at a time**.
2. **Slogan classifier** — an LSTM that predicts the industry of a business from its slogan.

The notebook follows the project requirements for preprocessing, model training, evaluation, and combining the two models. Both neural networks are trained for **50 epochs**.

The preprocessing explicitly creates the required `processed_slogan` and `modified_slogan` columns. The generator uses training subsequences so that each training example teaches the model to predict the **next word** rather than classify an entire slogan.


## 1. Project Setup

This section imports the libraries used throughout the notebook and fixes the random seeds so that the experiment is as reproducible as possible.

The project uses:

- **Pandas / NumPy** for data manipulation.
- **spaCy** for NLP preprocessing.
- **TensorFlow / Keras** for the LSTM models.
- **Scikit-learn** for the train/test split and classifier evaluation.


In [ ]:
# Install the spaCy English model if it is not already available.
# This cell is safe to run in Google Colab or a local Jupyter environment.

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("spacy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])

import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

print("spaCy version:", spacy.__version__)
print("spaCy English model loaded successfully.")


In [ ]:
# Import the remaining libraries.

import os
import random

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense,
    Concatenate,
    Flatten,
)
from tensorflow.keras.optimizers import Adam

# Set random seeds for reproducibility.
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


## 2. Loading the Slogan Dataset

The project dataset contains business information, including an `industry` column and an `output` column containing the slogan.

The task instructions require the relevant columns to be extracted and missing values to be handled before modelling.

The loader below searches common notebook locations for the slogan CSV. If it cannot find the file, update `DATA_PATH` to the location of the supplied dataset.


In [ ]:
# Locate the supplied slogan dataset.
# If your file has a different path, change DATA_PATH manually.

possible_paths = [
    "/content/slogan-valid (2).csv",
    "/content/slogan-valid (1).csv",
    "slogan-valid (2).csv",
    "slogan-valid (1).csv",
]

DATA_PATH = next(
    (path for path in possible_paths if os.path.exists(path)),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "The slogan CSV was not found. Upload the supplied slogan dataset "
        "to the notebook environment and update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset path:", DATA_PATH)
print("Dataset shape:", df.shape)

display(df.head())


## 3. Selecting Relevant Columns and Handling Missing Values

Only the fields required for this project are retained:

- `industry` — the target category.
- `output` — the slogan.

Rows with missing values in either field are removed. Text is converted to strings and surrounding whitespace is removed so that the preprocessing functions receive clean input.


In [ ]:
# Keep the fields required by the project.
# The company field is retained because the task also asks for business-name preprocessing.
required_columns = ["industry", "output"]

if "company" in df.columns:
    required_columns.append("company")

df = df[required_columns].copy()

# Remove rows where the modelling fields are missing.
df = df.dropna(subset=["industry", "output"]).copy()

# Standardise text values.
df["industry"] = df["industry"].astype(str).str.strip()
df["output"] = df["output"].astype(str).str.strip()

if "company" in df.columns:
    df["company"] = df["company"].fillna("").astype(str).str.strip()

# Remove rows that became empty after cleaning.
df = df[
    (df["industry"] != "") &
    (df["output"] != "")
].reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)
display(df.head())


## 4. spaCy Preprocessing

spaCy is used to preprocess the slogan and business-name text before tokenisation.

The required `processed_slogan` column contains a cleaned, lower-case sequence of alphabetic tokens. If the dataset contains a `company` column, the notebook also creates `processed_company`.

Stop-word removal is deliberately **not** used because short function words can be important to the structure and meaning of a slogan.

The required `modified_slogan` column adds explicit `<start>` and `<end>` markers. These markers tell the generator where a slogan begins and when it should stop generating.


In [ ]:
# Clean slogan text with spaCy.
def preprocess_text(text):
    doc = nlp(str(text).lower())

    tokens = [
        token.text
        for token in doc
        if token.is_alpha
    ]

    return " ".join(tokens)


# Create the required processed_slogan column.
df["processed_slogan"] = df["output"].apply(preprocess_text)

# Tokenise business names as requested when the company field is available.
if "company" in df.columns:
    df["processed_company"] = df["company"].apply(preprocess_text)

# Create the required modified_slogan column.
# The special tokens define the beginning and end of every slogan.
df["modified_slogan"] = (
    "<start> "
    + df["processed_slogan"]
    + " <end>"
)

# Remove slogans that contain no usable words after preprocessing.
df = df[df["processed_slogan"].str.strip() != ""].reset_index(drop=True)

preview_columns = [
    "industry",
    "output",
    "processed_slogan",
    "modified_slogan"
]

if "processed_company" in df.columns:
    preview_columns.insert(2, "processed_company")

display(df[preview_columns].head(10))


## 5. Creating the Industry-to-Index Dictionary

The generator receives the industry as a numerical input.

An explicit `industry_to_index` dictionary is therefore created to map every industry category to an integer. A reverse dictionary is also created so that predicted indices can be converted back into readable industry names.

The generator is trained using the industry index associated with each slogan.


In [ ]:
# Create a deterministic mapping between industry names and integer indices.
industries = sorted(df["industry"].unique())

industry_to_index = {
    industry: index
    for index, industry in enumerate(industries)
}

index_to_industry = {
    index: industry
    for industry, index in industry_to_index.items()
}

df["industry_index"] = df["industry"].map(industry_to_index)

print("Number of industries:", len(industry_to_index))
print("First five mappings:")

for industry, index in list(industry_to_index.items())[:5]:
    print(f"{industry!r} -> {index}")


## 6. Train/Test Split

The original slogan rows are split before creating generator subsequences.

This is important because splitting after subsequence creation could place different prefixes from the **same slogan** into both the training and testing sets, which would create data leakage.

A stratified 80/20 split keeps the distribution of industry categories as balanced as possible between the two sets.


In [ ]:
# Split the original rows first to prevent subsequence leakage.
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["industry_index"],
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))


## 7. Tokenising the Slogans

The Keras tokenizer converts the words in `modified_slogan` into integer token IDs.

The vocabulary includes the `<start>` and `<end>` tokens. These are necessary for generation because the model must learn both how to begin a slogan and when to terminate it.

The same tokenizer is used for training, classification, and generation so that the word-to-index mapping remains consistent.


In [ ]:
# Fit the tokenizer only on the training slogans.
tokenizer = Tokenizer(
    lower=True,
    filters='',
    oov_token="<unk>"
)

tokenizer.fit_on_texts(train_df["modified_slogan"])

word_to_index = tokenizer.word_index
index_to_word = {
    index: word
    for word, index in word_to_index.items()
}

vocab_size = len(word_to_index) + 1

# Convert slogans to token sequences.
train_sequences = tokenizer.texts_to_sequences(
    train_df["modified_slogan"]
)

test_sequences = tokenizer.texts_to_sequences(
    test_df["modified_slogan"]
)

max_length = max(
    max(len(sequence) for sequence in train_sequences),
    max(len(sequence) for sequence in test_sequences)
)

print("Vocabulary size:", vocab_size)
print("Maximum slogan length:", max_length)


## 8. Building Generator Subsequences

The generator must learn to predict the **next word**, not classify a complete slogan.

For example, a training slogan is conceptually transformed into prefix/target pairs:

```text
<start>                 -> first word
<start> first           -> second word
<start> first second   -> third word
...
```

Each prefix is padded to the same length. The industry index is repeated for every prefix belonging to that slogan.

This creates the supervised training data required for one-word-at-a-time generation.


In [ ]:
# Create prefix/next-word training examples.
def create_generator_subsequences(dataframe, sequences):
    prefix_sequences = []
    industry_indices = []
    next_words = []

    for row_index, sequence in enumerate(sequences):
        industry_index = int(
            dataframe.iloc[row_index]["industry_index"]
        )

        # Predict every token after the first token.
        for position in range(1, len(sequence)):
            prefix = sequence[:position]
            target = sequence[position]

            prefix_sequences.append(prefix)
            industry_indices.append(industry_index)
            next_words.append(target)

    prefix_sequences = pad_sequences(
        prefix_sequences,
        maxlen=max_length - 1,
        padding="pre"
    )

    return (
        np.array(prefix_sequences, dtype=np.int32),
        np.array(industry_indices, dtype=np.int32),
        np.array(next_words, dtype=np.int32),
    )


X_gen_train, industry_gen_train, y_gen_train = (
    create_generator_subsequences(
        train_df,
        train_sequences
    )
)

X_gen_test, industry_gen_test, y_gen_test = (
    create_generator_subsequences(
        test_df,
        test_sequences
    )
)

print("Generator prefix shape:", X_gen_train.shape)
print("Generator industry shape:", industry_gen_train.shape)
print("Generator target shape:", y_gen_train.shape)


## 9. Building the LSTM Slogan Generator

The generator uses two inputs:

1. An **industry index**, which tells the model which business category the slogan should represent.
2. A **partial slogan**, which contains the words generated so far.

The two representations are combined before the final Dense layer predicts the probability of every word in the vocabulary.

This architecture directly addresses the project requirement that the generator use the industry category as an input and generate the slogan sequentially.


In [ ]:
# Industry input.
industry_input = Input(
    shape=(1,),
    name="industry_input"
)

industry_embedding = Embedding(
    input_dim=len(industry_to_index),
    output_dim=32,
    name="industry_embedding"
)(industry_input)

industry_embedding = Flatten()(industry_embedding)

# Partial slogan input.
sequence_input = Input(
    shape=(max_length - 1,),
    name="sequence_input"
)

sequence_embedding = Embedding(
    input_dim=vocab_size,
    output_dim=100,
    mask_zero=True,
    name="word_embedding"
)(sequence_input)

sequence_lstm = LSTM(
    128,
    name="generator_lstm"
)(sequence_embedding)

# Combine the industry representation with the partial slogan.
combined = Concatenate()([
    sequence_lstm,
    industry_embedding
])

next_word_output = Dense(
    vocab_size,
    activation="softmax",
    name="next_word_output"
)(combined)

gen_model = Model(
    inputs=[industry_input, sequence_input],
    outputs=next_word_output,
    name="slogan_generator"
)

gen_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

gen_model.summary()


## 10. Training the Slogan Generator

The generator is trained for the required **50 epochs**.

Each training example contains a partial slogan and its industry, while the target is the next word.

Therefore, the model is explicitly learning the next-word prediction task required for sequential slogan generation.


In [ ]:
# Train the generator for the required 50 epochs.
gen_history = gen_model.fit(
    [industry_gen_train, X_gen_train],
    y_gen_train,
    validation_data=(
        [industry_gen_test, X_gen_test],
        y_gen_test
    ),
    epochs=50,
    batch_size=64,
    verbose=1
)


## 11. Generating a Slogan One Word at a Time

The `generate_slogan()` function accepts an **industry name** rather than a slogan.

Generation begins with the `<start>` token. The model predicts one word, that predicted word is appended to the sequence, and the expanded sequence is passed back to the model.

The process continues until the model predicts `<end>` or the maximum slogan length is reached.

No manually written slogan examples are used.


In [ ]:
# Helper function for case-insensitive industry lookup.
def get_industry_index(industry_name):
    normalised_name = str(industry_name).strip().lower()

    for industry, index in industry_to_index.items():
        if industry.lower() == normalised_name:
            return index

    raise ValueError(
        "Unknown industry. Choose an industry from "
        "industry_to_index.keys()."
    )


def generate_slogan(industry_name, max_words=None):
    """Generate a slogan one word at a time for an industry."""

    industry_index = get_industry_index(industry_name)

    start_token = tokenizer.word_index["<start>"]
    end_token = tokenizer.word_index["<end>"]

    # The maximum number of generated words is based on the
    # maximum length observed during training.
    if max_words is None:
        max_words = max_length - 2

    generated_tokens = [start_token]

    for _ in range(max_words):

        # Keep the generated prefix and pad it to the model input size.
        prefix = pad_sequences(
            [generated_tokens],
            maxlen=max_length - 1,
            padding="pre"
        )

        industry_array = np.array(
            [[industry_index]],
            dtype=np.int32
        )

        # Predict the next word.
        probabilities = gen_model.predict(
            [industry_array, prefix],
            verbose=0
        )[0]

        next_token = int(
            np.argmax(probabilities)
        )

        # Stop if the model predicts the end token.
        if next_token == end_token:
            break

        # Avoid adding padding or unknown control tokens.
        if next_token == 0:
            break

        generated_tokens.append(next_token)

    generated_words = [
        index_to_word.get(token, "")
        for token in generated_tokens[1:]
    ]

    generated_words = [
        word for word in generated_words
        if word not in {"<start>", "<end>", "<unk>"}
    ]

    return " ".join(generated_words).strip()


## 12. Testing the Generator Without Hand-Written Slogans

The generator is tested using industries that actually occur in the supplied dataset.

This keeps the test inputs tied to the data rather than introducing manually invented slogan examples.


In [ ]:
# Select industries directly from the dataset.
test_industries = list(
    industry_to_index.keys()
)[:5]

for industry in test_industries:
    generated = generate_slogan(industry)

    print(f"Industry: {industry}")
    print(f"Generated slogan: {generated}")
    print("-" * 70)


## 13. Building the Slogan Classifier

The classifier performs the reverse task.

It receives a tokenised slogan and predicts the industry category associated with that slogan.

The classifier uses:

- an Embedding layer,
- an LSTM layer,
- a Dense softmax output layer.

The output contains a probability for every industry in the training dataset.


In [ ]:
# Prepare padded slogan inputs for classification.
X_classifier_train = pad_sequences(
    train_sequences,
    maxlen=max_length,
    padding="post"
)

X_classifier_test = pad_sequences(
    test_sequences,
    maxlen=max_length,
    padding="post"
)

y_classifier_train = train_df["industry_index"].to_numpy(
    dtype=np.int32
)

y_classifier_test = test_df["industry_index"].to_numpy(
    dtype=np.int32
)

classifier = Sequential(
    [
        Embedding(
            input_dim=vocab_size,
            output_dim=100,
            mask_zero=True
        ),
        LSTM(128),
        Dense(
            len(industry_to_index),
            activation="softmax"
        )
    ],
    name="slogan_classifier"
)

classifier.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

classifier.summary()


## 14. Training the Slogan Classifier

The classifier is trained for the required **50 epochs**.

The model learns to associate the words in a slogan with the corresponding industry category.

The test set is kept separate from the training data and is used for evaluation after training.


In [ ]:
# Train the classifier for the required 50 epochs.
classifier_history = classifier.fit(
    X_classifier_train,
    y_classifier_train,
    validation_data=(
        X_classifier_test,
        y_classifier_test
    ),
    epochs=50,
    batch_size=32,
    verbose=1
)


## 15. Evaluating the Slogan Classifier

The classifier is evaluated using accuracy.

Accuracy is the proportion of test slogans for which the predicted industry matches the actual industry.

A classification report is also produced so that performance can be inspected across individual industry categories.


In [ ]:
# Evaluate the classifier on unseen test slogans.
test_loss, test_accuracy = classifier.evaluate(
    X_classifier_test,
    y_classifier_test,
    verbose=0
)

classifier_probabilities = classifier.predict(
    X_classifier_test,
    verbose=0
)

classifier_predictions = np.argmax(
    classifier_probabilities,
    axis=1
)

overall_accuracy = accuracy_score(
    y_classifier_test,
    classifier_predictions
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Accuracy score: {overall_accuracy:.4f}")


## 16. Creating `classify_slogan()`

The `classify_slogan()` function accepts a raw slogan from the user.

It performs the same preprocessing used during training, converts the slogan into token IDs, pads the sequence, and passes it through the trained classifier.

The predicted numerical class is then converted back into the corresponding industry name.


In [ ]:
def classify_slogan(slogan):
    """Predict the industry associated with a slogan."""

    processed = preprocess_text(slogan)

    sequence = tokenizer.texts_to_sequences(
        [f"<start> {processed} <end>"]
    )

    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post"
    )

    probabilities = classifier.predict(
        padded,
        verbose=0
    )[0]

    predicted_index = int(
        np.argmax(probabilities)
    )

    predicted_industry = index_to_industry[
        predicted_index
    ]

    confidence = float(
        probabilities[predicted_index]
    )

    return predicted_industry, confidence


## 17. Combining the Generator and Classifier

The final part of the project connects both models.

For an industry taken from the dataset:

1. The generator creates a new slogan one word at a time.
2. The generated slogan is passed to the classifier.
3. The classifier predicts the industry represented by the generated slogan.
4. The requested industry and predicted industry are compared.

This directly tests whether the generator produces text that preserves enough industry-specific information for the classifier to recognise it.


In [ ]:
# Run the generator -> classifier pipeline using industries
# selected directly from the dataset.

combined_results = []

for industry in test_industries:

    generated_slogan = generate_slogan(industry)

    if generated_slogan:
        predicted_industry, confidence = classify_slogan(
            generated_slogan
        )
    else:
        predicted_industry = "No slogan generated"
        confidence = 0.0

    combined_results.append(
        {
            "requested_industry": industry,
            "generated_slogan": generated_slogan,
            "classifier_prediction": predicted_industry,
            "classifier_confidence": confidence,
            "match": industry.lower() == predicted_industry.lower()
        }
    )

combined_results = pd.DataFrame(combined_results)

display(combined_results)


## 18. Generator/Classifier Results

The table above provides an empirical comparison between the industry supplied to the generator and the industry predicted from the generated slogan.

A match means the classifier assigned the generated slogan to the same industry requested from the generator.

The generator and classifier solve different problems, so a mismatch does not automatically mean that either model is incorrect. A generated slogan may be linguistically plausible while still containing wording that overlaps with another industry's vocabulary.


In [ ]:
# Summarise the combined-model experiment.
if len(combined_results) > 0:
    pipeline_match_rate = combined_results["match"].mean()

    print(
        "Generator -> classifier match rate: "
        f"{pipeline_match_rate:.2%}"
    )

    display(
        combined_results[
            [
                "requested_industry",
                "generated_slogan",
                "classifier_prediction",
                "classifier_confidence",
                "match",
            ]
        ]
    )


## 19. Final Results and Conclusions

The final conclusions should be based on the values produced by the completed 50-epoch training runs.

The notebook reports:

- classifier test accuracy,
- generator next-word validation performance,
- generated slogans for dataset industries,
- classifier predictions for generated slogans,
- and the generator-to-classifier match rate.

The main limitation is the structure of the dataset: industries can have very different numbers of examples, and the generator is learning from a relatively small collection of short slogans. Consequently, generated text may be repetitive or semantically ambiguous.

Most importantly, the generator is now a genuine sequential model: it is conditioned on the industry and predicts **one next word at a time**. The classifier independently predicts an industry from a slogan. This makes the final comparison between the two models meaningful.


In [ ]:
# Final numerical summary.
print("=" * 70)
print("FINAL MODEL RESULTS")
print("=" * 70)

print(f"Classifier test accuracy: {test_accuracy:.4f}")

if len(gen_history.history["val_accuracy"]) > 0:
    print(
        "Generator final validation accuracy: "
        f"{gen_history.history['val_accuracy'][-1]:.4f}"
    )

if "pipeline_match_rate" in globals():
    print(
        "Generator -> classifier match rate: "
        f"{pipeline_match_rate:.2%}"
    )
